ESERCIZIO

Calcoliamo Similitudine Coseno tra Testo e Immagine
1) utilizza un modello CLIP pre-addestrato per estrarre l'embedding di un'immagine di un gatto e di tre frasi: "un gatto", "un cane", "un tramonto"
2) calcla il prodotto scalare tra il vettore immagine e i tre vettori testo
3) identifica quale frase ottiene lo score più alto e spiega perchè la normalizzazione è fondamentale in questo processo

Suggerimento: usa 'torch.nn.functional_cosine_similarity' per il calcolo

In [ ]:
# Testo ed Immagini vengono porti nello stesso spazio vettoriale.

import torch
import torch.nn.functional as F
from PIL import Image
import requests
from transformers import CLIPProcessor, CLIPModel

# 1. Setup e caricamento del modello CLIP
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id) #scarica la configurazione dei pesi già addestrati

# Prepariamo l'immagine del gatto e le tre frasi
url = "http://images.cocodataset.org/val2017/000000039769.jpg" # Foto di gatti
image = Image.open(requests.get(url, stream=True).raw) # carico l'immagine
labels = ["un gatto", "un cane", "un tramonto"] #CLIP riceve label in linguaggio naturale, non riceve lebel numeriche

# 2. Estrazione degli Embedding
# preparo immagini e testo nel formato CLIP
inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)

with torch.no_grad():
    outputs = model(**inputs)
    # CLIP restituisce già i vettori proiettati nello spazio condiviso
    image_embeds = outputs.image_embeds  # Shape: [1, 512]
    text_embeds = outputs.text_embeds    # Shape: [3, 512]

# 3. Calcolo della Similitudine Coseno
# Usiamo torch.nn.functional per calcolare la similitudine tra l'immagine e ogni testo
# Espandiamo il vettore immagine per confrontarlo con tutti i testi contemporaneamente
similarities = F.cosine_similarity(image_embeds, text_embeds)

# Risultati
print(f"Risultati della similitudine:")
for label, score in zip(labels, similarities):
    print(f" - {label}: {score.item():.4f}")

# Identificazione del vincitore
best_idx = similarities.argmax().item()
print(f"\nLa frase con lo score più alto è: '{labels[best_idx]}'")